# Pancancer enrichment analysis step 3: Make figures from Reactome data

## Setup

In [1]:
import cptac
import cptac.utils as ut
import os
import datetime
import altair as alt
import numpy as np
import pandas as pd

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

STEP1_DIR = "step1_outputs"
STEP1_FILE = "tumor_change_20200529_195104_10000_perm.tsv"
STEP1_FILE_PATH = os.path.join(STEP1_DIR, STEP1_FILE)

STEP2_DIR = "step2_outputs"
STEP2_FILE = "enrichment_20200609_171630_from_tumor_change_20200529_195104_10000_perm.tsv"
STEP2_FILE_PATH = os.path.join(STEP2_DIR, STEP2_FILE)

## Read in data from steps 1 and 2

In [3]:
expression_data = pd.read_csv(STEP1_FILE_PATH, sep="\t", index_col=0)

# Select just the proteins where we chose to reject the null hypothesis of no change
expression_data = expression_data[expression_data["reject_null"]]

# Make a column where all increases are +1 and all decreases 
# are -1, because these are ratioed abundances, so we can't 
# compare magnitudes between genes--we can only compare whether 
# there was a change, and whether it was positive or negative
expression_data = expression_data.assign(simplified_change=expression_data["change"])

expression_data.loc[expression_data["change"] > 0, "simplified_change"] = 1
expression_data.loc[expression_data["change"] < 0, "simplified_change"] = -1
expression_data.loc[expression_data["change"] == 0, "simplified_change"] = 0

expression_data

,cancer_type,protein,change,P_val,adj_p,reject_null,protein_str,simplified_change
0,ccrcc,"('A1BG', 'NP_570602.2')",0.282928,0.0000,0.000000,True,A1BG,1.0
1,ccrcc,"('A1CF', 'NP_620310.1')",-0.551358,0.0000,0.000000,True,A1CF,-1.0
2,ccrcc,"('A2M', 'NP_000005.2')",0.595512,0.0000,0.000000,True,A2M,1.0
4,ccrcc,"('AAAS', 'NP_056480.1')",0.173579,0.0000,0.000000,True,AAAS,1.0
5,ccrcc,"('AACS', 'NP_076417.2')",-0.215967,0.0000,0.000000,True,AACS,-1.0
7,ccrcc,"('AADAT', 'NP_001273612.1')",-1.818291,0.0000,0.000000,True,AADAT,-1.0
8,ccrcc,"('AAED1', 'NP_714542.1')",0.426052,0.0000,0.000000,True,AAED1,1.0
9,ccrcc,"('AAGAB', 'NP_078942.3')",-0.070117,0.0009,0.001554,True,AAGAB,-1.0
10,ccrcc,"('AAK1', 'NP_055726.3')",-0.307917,0.0000,0.000000,True,AAK1,-1.0
11,ccrcc,"('AAMP', 'NP_001078.2')",0.328956,0.0000,0.000000,True,AAMP,1.0


In [4]:
enrichment_data = pd.read_csv(STEP2_FILE_PATH, sep="\t", index_col=0)
enrichment_data

,pathway_id,cancer_type,mean_expr,url,name,pathway_times_enriched,pathway_avg_rank,entities_pValue,entities_fdr,cancer_rank_pval
0,R-HSA-6798695,ccrcc,0.233503,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,1.000,3.467008e-03,8.426404e-01,1.0
1,R-HSA-6798695,colon,-0.134831,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,1.000,6.216916e-12,1.500142e-08,1.0
2,R-HSA-6798695,endometrial,0.288515,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,1.000,1.403496e-03,8.219205e-01,1.0
3,R-HSA-6798695,gbm,0.166227,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,1.000,2.129677e-03,8.037977e-01,1.0
4,R-HSA-6798695,hnscc,0.133159,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,1.000,3.649260e-03,8.329209e-01,1.0
5,R-HSA-6798695,lscc,-0.407595,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,1.000,1.637558e-03,8.726502e-01,1.0
6,R-HSA-6798695,luad,-0.410000,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,1.000,1.800932e-04,4.412282e-01,1.0
7,R-HSA-6798695,ovarian,-0.303754,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,1.000,3.532076e-05,8.660650e-02,1.0
8,R-HSA-6791226,ccrcc,0.699422,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Major pathway of rRNA processing in the nucleo...,8,2.875,3.132093e-02,8.426404e-01,4.0
9,R-HSA-6791226,colon,0.771429,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Major pathway of rRNA processing in the nucleo...,8,2.875,2.825057e-07,1.376497e-04,2.0


## Figure 1: Bubble plot

In [5]:
enrichment_data = enrichment_data.assign(
    rank_size=1 / enrichment_data["cancer_rank_pval"],
    avg_rank_size=1 / enrichment_data["pathway_avg_rank"],
    avg_rank_label=enrichment_data["pathway_avg_rank"].astype(str))

# Take care of duplicates for the upper plot
# If there were more than two values that were the same, we'd 
# need to repeat this multiple times until there was no change.

enrichment_data["avg_rank_label"] = enrichment_data["avg_rank_label"].where(
    cond=~(enrichment_data.duplicated(subset=["cancer_type", "avg_rank_label"], keep="first")),
    other=" " + enrichment_data["avg_rank_label"])

In [6]:
individual = alt.Chart(enrichment_data).mark_circle().encode(
    x=alt.X(
        "name:N",
        sort=enrichment_data["name"].values,
        axis=alt.Axis(
            labelAngle=-30,
            labelFontSize=12,
            labelLimit=500,
            title="",
            titleFontSize=12
        )
    ),
    y=alt.Y(
        "cancer_type:N",
        axis=alt.Axis(
            title="Cancer type",
            titleFontSize=12
        ),
    ),
    size=alt.Size(
        "rank_size:Q",
        legend=alt.Legend(
            title="1 / rank"
        )
    ),
    color=alt.Color(
        "mean_expr:Q",
        scale=alt.Scale(
            scheme="blueorange"
        ),
        legend=alt.Legend(
            title="Pathway tumor expression"
        )
    )
).properties(
    width=400,
    height=300
)

aggregate = alt.Chart(enrichment_data).mark_circle().encode(
    x=alt.X(
        "avg_rank_label:N",
        sort=enrichment_data["avg_rank_label"].values,
        axis=alt.Axis(
            labelAngle=-30,
            labelFontSize=12,
            labelLimit=500,
            title="Overall rank of pathway",
            titleFontSize=12
        )
    ),
    size=alt.Size(
        "avg_rank_size:Q",
        legend=alt.Legend(
            title="1 / rank"
        )
    ),
).properties(
    width=400
)

alt.vconcat(aggregate, individual).configure_axis(
    grid=True
)

alt.VConcatChart(...)

## Figure 2: Expression of neutrophil degranulation pathway proteins

In [7]:
neutro_pathway_id = enrichment_data.\
    loc[enrichment_data["name"] == "Neutrophil degranulation", "pathway_id"].\
    unique()

if len(neutro_pathway_id) != 1:
    raise ValueError("Multiple pathways found.")
    
neutro_pathway_id = neutro_pathway_id[0]

neutro_proteins = ut.search_reactome_proteins_in_pathways(neutro_pathway_id)
prot_expr = expression_data[expression_data["protein_str"].isin(neutro_proteins["member"])]

neutro_tumor_up = enrichment_data.\
    loc[(enrichment_data["pathway_id"] == neutro_pathway_id) & (enrichment_data["mean_expr"] > 0),
       "cancer_type"]

neutro_tumor_down = enrichment_data.\
    loc[(enrichment_data["pathway_id"] == neutro_pathway_id) & (enrichment_data["mean_expr"] < 0),
       "cancer_type"]

def barplot_expr(expr_data, cancer_types):
    cancer_expr = expr_data[expr_data["cancer_type"].isin(cancer_types)].\
        groupby("protein_str")["simplified_change"].agg(np.mean).\
        reset_index(drop=False).\
        sort_values(by=["simplified_change", "protein_str"])
    
    plot = alt.Chart(cancer_expr).mark_bar().encode(
        x=alt.X(
            "protein_str:N",
            sort=cancer_expr["protein_str"].values,
            axis=alt.Axis(
                title="Proteins",
                titleFontSize=14,
                labels=False,
                ticks=False
            )
        ),
        y=alt.Y(
            "simplified_change:Q",
            axis=alt.Axis(
                title="Mean expression across cancers",
                titleFontSize=14
            )
        ),
        color=alt.condition(
            alt.datum.simplified_change > 0,
            alt.value("#e45756"),
            alt.value("#4c78a8")
        )
    ).properties(
        width=600,
        height=500
    )
    
    return plot

In [8]:
alt.vconcat(
    barplot_expr(prot_expr, neutro_tumor_up).properties(
        title="Cancers with pathway overall up in tumor"
    ),
    barplot_expr(prot_expr, neutro_tumor_down).properties(
        title="Cancers with pathway overall down in tumor"
    )
).configure_title(
    fontSize=18
)

alt.VConcatChart(...)

## Figure 3: Distribution of tumor stage in each cancer

In [9]:
cc = cptac.Ccrcc(no_internet=True)
co = cptac.Colon(no_internet=True)
en = cptac.Endometrial(no_internet=True)
gb = cptac.Gbm(no_internet=True)
hn = cptac.Hnscc(no_internet=True)
ls = cptac.Lscc(no_internet=True)
lu = cptac.Luad(no_internet=True)
ov = cptac.Ovarian(no_internet=True)

cptac warning: The GBM dataset is under publication embargo until March 01, 2021. CPTAC is a community resource project and data are made available rapidly after generation for community research use. The embargo allows exploring and utilizing the data, but analysis may not be published until after the embargo date. Please see https://proteomics.cancer.gov/data-portal/about/data-use-agreement or enter cptac.embargo() to open the webpage for more details. (/home/caleb/anaconda3/envs/dev/lib/python3.7/site-packages/ipykernel_launcher.py, line 4)


cptac warning: The HNSCC data is currently strictly reserved for CPTAC investigators. Otherwise, you are not authorized to access these data. Additionally, even after these data become publicly available, they will be subject to a publication embargo (see https://proteomics.cancer.gov/data-portal/about/data-use-agreement or enter cptac.embargo() to open the webpage for more details). (/home/caleb/anaconda3/envs/dev/lib/python3.7/site-packages/ipykernel_launcher.py, line 5)


cptac warning: The LSCC data is currently strictly reserved for CPTAC investigators. Otherwise, you are not authorized to access these data. Additionally, even after these data become publicly available, they will be subject to a publication embargo (see https://proteomics.cancer.gov/data-portal/about/data-use-agreement or enter cptac.embargo() to open the webpage for more details). (/home/caleb/anaconda3/envs/dev/lib/python3.7/site-packages/ipykernel_launcher.py, line 6)


cptac warning: The LUAD dataset is under publication embargo until July 01, 2020. CPTAC is a community resource project and data are made available rapidly after generation for community research use. The embargo allows exploring and utilizing the data, but analysis may not be published until after the embargo date. Please see https://proteomics.cancer.gov/data-portal/about/data-use-agreement or enter cptac.embargo() to open the webpage for more details. (/home/caleb/anaconda3/envs/dev/lib/python3.7/site-packages/ipykernel_launcher.py, line 7)


In [10]:
stage_df = pd.DataFrame()

In [11]:
cc_stage = cc.get_clinical()["tumor_stage_pathological"].\
    dropna().\
    str.rsplit(" ", n=1, expand=True)[1].\
    rename("tumor_stage").\
    reset_index(drop=False).\
    assign(cancer_type="ccrcc")

stage_df = stage_df.append(cc_stage)

In [12]:
co_stage = co.get_clinical()["Stage"].\
    dropna().\
    str.rsplit(" ", n=1, expand=True)[1].\
    rename("tumor_stage").\
    reset_index(drop=False).\
    assign(cancer_type="colon")

stage_df = stage_df.append(co_stage)

In [13]:
en_stage = en.get_clinical()["tumor_Stage-Pathological"].\
    dropna().\
    str.rsplit(" ", n=1, expand=True)[1].\
    rename("tumor_stage").\
    reset_index(drop=False).\
    assign(cancer_type="endometrial")

stage_df = stage_df.append(en_stage)

In [14]:
# GBM?

In [15]:
hn_stage = hn.get_clinical()["patho_staging_curated"].\
    dropna().\
    str.rsplit(" ", n=1, expand=True)[1].\
    rename("tumor_stage").\
    reset_index(drop=False).\
    assign(cancer_type="hnscc")

stage_df = stage_df.append(hn_stage)

In [16]:
ls_stage = ls.get_clinical()["Stage"]
ls_stage = ls_stage.\
    rename("tumor_stage").\
    where(
        cond=~(ls_stage.dropna().str.endswith("A") | ls_stage.dropna().str.endswith("B")),
        other=ls_stage.str[:-1]).\
    where(
        cond=~(ls_stage.dropna().str.endswith("A 3")),
        other=ls_stage.dropna().str[:-3]
    ).\
    reset_index(drop=False).\
    assign(cancer_type="lscc")

stage_df = stage_df.append(ls_stage)

In [17]:
lu_stage = lu.get_clinical()["Stage"]
lu_stage = lu_stage.\
    rename("tumor_stage").\
    where(
        cond=~(lu_stage.dropna().str.endswith("A") | lu_stage.dropna().str.endswith("B")),
        other=lu_stage.str[:-1]).\
    replace({
        "1": "I",
        "2": "II",
        "3": "III"}).\
    reset_index(drop=False).\
    assign(cancer_type="luad")

stage_df = stage_df.append(lu_stage)

In [18]:
ov_stage = ov.get_clinical()["Tumor_Stage_Ovary_FIGO"]
ov_stage = ov_stage.\
    rename("tumor_stage").\
    where(
        cond=~(ov_stage.dropna().str.endswith("A") | 
               ov_stage.dropna().str.endswith("B") | 
               ov_stage.dropna().str.endswith("C")),
        other=ov_stage.str[:-1]).\
    drop(ov_stage[ov_stage == "Not Reported/ Unknown"].index).\
    reset_index(drop=False).\
    assign(cancer_type="ovarian")

stage_df = stage_df.append(ov_stage)

In [19]:
stage_df = stage_df.dropna()

In [20]:
# Join in whether neutrophil expression is up or down in tumor
neutro_expr = enrichment_data.loc[
    enrichment_data["pathway_id"] == neutro_pathway_id, 
    ["cancer_type", "name", "pathway_id", "mean_expr"]]

neutro_expr = neutro_expr.assign(
    neutro_expr_tumor=np.where(neutro_expr["mean_expr"] > 0, "Up", "Down"))


neutro_expr

,cancer_type,name,pathway_id,mean_expr,neutro_expr_tumor
0,ccrcc,Neutrophil degranulation,R-HSA-6798695,0.233503,Up
1,colon,Neutrophil degranulation,R-HSA-6798695,-0.134831,Down
2,endometrial,Neutrophil degranulation,R-HSA-6798695,0.288515,Up
3,gbm,Neutrophil degranulation,R-HSA-6798695,0.166227,Up
4,hnscc,Neutrophil degranulation,R-HSA-6798695,0.133159,Up
5,lscc,Neutrophil degranulation,R-HSA-6798695,-0.407595,Down
6,luad,Neutrophil degranulation,R-HSA-6798695,-0.410000,Down
7,ovarian,Neutrophil degranulation,R-HSA-6798695,-0.303754,Down


In [21]:
stage_df = stage_df.merge(
    right=neutro_expr,
    how="left",
    left_on="cancer_type",
    right_on="cancer_type",
    validate="many_to_one")

In [22]:
# Make plots
alt.Chart(stage_df).mark_bar().encode(
    x=alt.X(
        "tumor_stage:N",
        axis=alt.Axis(
            labelAngle=0,
            labelFontSize=14,
            title=""
        )
    ),
    y=alt.Y(
        "count():Q",
        axis=alt.Axis(
            title="Count",
            titleFontSize=14
        )
    ),
    color=alt.Color(
        "neutro_expr_tumor",
        legend=alt.Legend(
            title=["Neutrophil degradation:", "Mean pathway expression", "in tumor"]
        ),
        scale=alt.Scale(
            domain=["Up", "Down"],
            range=["#e45756", "#4c78a8"]
        )
    ),
    column=alt.Column(
        "cancer_type:N",
        title="Tumor stage distribution"
    )
).configure_title(
    fontSize=16
)

alt.Chart(...)

In [23]:
# Ask Sam: Is this idea of "mean pathway expression" 
# really valid? What if in reality, though we can't
# measure it, some proteins' magnitude of expression 
# change is a lot greater, so they should actually
# have greater weight in the mean calculation, and
# thus might alter the overall statistic for the
# pathway?